# Cell-1: Utilities & Core Ops — Parameters & Behavior

**Purpose**  
Common GPU/CPU backend, gradient/div operators, FFT symbols, simple filters, and I/O (2D/3D stack). Designed for 16 GB VRAM, float32.

**Core helpers**  
- `get_xp(use_gpu)` → choose `cupy` or `numpy`  
- `grad_nd(u, bc)` / `div_nd(p, bc)` → forward gradient & matching divergence (Neumann/Periodic)  
- `fft_freq_symbol` / `fft_denominator` → frequency symbols for FFT solvers (periodic)  
- `avg_filter_nd` → small-radius box blur (no SciPy) for guidance/tensors  
- `read_image_2d` / `read_stack_3d` → input helpers (normalize to [0,1], float32)

**BC notes**  
- Spatial solvers (TV/TGV/PnP/etc.): prefer **Neumann** (replicated)  
- FFT solvers (L0/L2): inherently **Periodic** → possible wrap at borders

**Tips**  
- Keep **float32** to stay memory-light.  
- If 3D FFT OOM: switch to TV/TGV/PnP (and slab Z for 3D).



In [5]:
# === Utilities & Core Ops ===
import os, re, math, time
import numpy as np
import imageio.v3 as iio

try:
    import cupy as cp
    _GPU_OK = True
except Exception:
    cp = None
    _GPU_OK = False

def get_xp(use_gpu: bool = True):
    return cp if (_GPU_OK and use_gpu) else np

def to_xp(a, xp):
    return a if (xp is np or isinstance(a, np.ndarray)) else cp.asarray(a)

def to_cpu(a):
    return a if isinstance(a, np.ndarray) else cp.asnumpy(a)

def normalize_float(img, dtype='float32'):
    if img.dtype.kind in 'ui':
        img = img.astype(dtype, copy=False) / np.iinfo(img.dtype).max
    else:
        img = img.astype(dtype, copy=False); img = np.clip(img, 0, 1)
    return img

def grad_nd(u, bc='neumann'):
    xp = cp.get_array_module(u) if (cp is not None and isinstance(u, cp.ndarray)) else np
    grads = []
    for ax in range(u.ndim):
        g = xp.zeros_like(u)
        sls = [slice(None)]*u.ndim
        slt = [slice(None)]*u.ndim
        sls[ax] = slice(1, None); slt[ax] = slice(0, -1)
        g[tuple(slt)] = u[tuple(sls)] - u[tuple(slt)]
        if bc == 'periodic':
            last = [slice(None)]*u.ndim; last[ax] = -1
            first= [slice(None)]*u.ndim; first[ax]= 0
            g[tuple(last)] = u[tuple(first)] - u[tuple(last)]
        grads.append(g)
    return grads

def div_nd(ps, bc='neumann'):
    p0 = ps[0]
    xp = cp.get_array_module(p0) if (cp is not None and isinstance(p0, cp.ndarray)) else np
    out = xp.zeros_like(p0)
    for ax,p in enumerate(ps):
        sl = [slice(None)]*p.ndim; sl[ax]=slice(1,None)
        d = xp.zeros_like(p)
        d[tuple(sl)] = p[tuple(sl)] - p[tuple([slice(None) if i!=ax else slice(0,-1) for i in range(p.ndim)])]
        out += d
        if bc=='periodic':
            first=[slice(None)]*p.ndim; first[ax]=0
            last =[slice(None)]*p.ndim; last [ax]=-1
            out[tuple(first)] += p[tuple(first)] - p[tuple(last)]
    return out

def fft_freq_symbol(shape, xp):
    axes=[]
    for n in shape:
        k = xp.arange(n, dtype=xp.float32)
        w = 2*xp.pi*k/float(n)
        axes.append(4.0*xp.sin(0.5*w)**2)
    return xp.meshgrid(*axes, indexing='ij')

def fft_denominator(shape, alpha, xp):
    grids = fft_freq_symbol(shape, xp)
    S = grids[0]
    for g in grids[1:]: S = S + g
    return 1.0 + alpha*S

# simple nD box blur (radius r -> kernel size 2r+1)
def avg_filter_nd(a, r=1):
    xp = cp.get_array_module(a) if (cp is not None and isinstance(a, cp.ndarray)) else np
    out = xp.zeros_like(a)
    cnt = 0
    shifts = [range(-r, r+1)]*a.ndim
    from itertools import product
    for offs in product(*shifts):
        aa = a
        for ax,sh in enumerate(offs):
            aa = xp.roll(aa, sh, axis=ax)
        out += aa; cnt += 1
    return out / float(cnt)

# I/O helpers
_num_extract = re.compile(r'(\d+)')
def _sort_natural(files):
    def key(s):
        parts = _num_extract.split(s)
        return [int(p) if p.isdigit() else p for p in parts]
    return sorted(files, key=key)

def read_image_2d(path, use_gpu=True, dtype='float32'):
    img = iio.imread(path)
    if img.ndim==3: img = img.mean(axis=-1)
    img = normalize_float(img, dtype)
    xp = get_xp(use_gpu); return to_xp(img, xp)

def read_stack_3d(folder, use_gpu=True, dtype='float32',
                  exts=('.tif','.tiff','.png','.jpg','.jpeg')):
    files = [f for f in os.listdir(folder) if f.lower().endswith(exts)]
    if not files: raise FileNotFoundError("No slices in folder.")
    files = _sort_natural(files)
    arrs = [normalize_float(iio.imread(os.path.join(folder,f)), dtype) for f in files]
    vol = np.stack([a if a.ndim==2 else a.mean(axis=-1) for a in arrs], axis=0) # Z,Y,X
    xp = get_xp(use_gpu); return to_xp(vol, xp)


# Base Models — TV / L0 / L2 / L3 / TGV² / PnP-ADMM (TV)

**Includes**  
- **TV (L1 on ∇u)**: Primal–Dual (default), Chambolle, Split-Bregman  
- **L2 on ∇u**: FFT closed form (periodic)  
- **L3 on ∇u**: explicit stabilized descent  
- **L0 on ∇u**: Half-quadratic splitting + FFT (periodic)  
- **TGV²**: second-order TV via primal–dual (single-λ: α₁=λ, α₀=2λ)  
- **PnP-ADMM (TV)**: λ-only UI (ρ=1 fixed, small inner TV iters)

**Parameters**  
- `lam` / `alpha` strength; `max_iter`, `step` for iterative methods; `bc` (Neumann/Periodic)


In [6]:
# === Base Models ===
def tv_primal_dual(f, lam=0.1, max_iter=300, theta=1.0, isotropic=True, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u = f.copy(); u_bar = u.copy()
    ps = [xp.zeros_like(f) for _ in range(f.ndim)]
    L2 = 4.0*f.ndim; tau = math.sqrt(0.9/L2); sigma = math.sqrt(0.9/L2)
    for _ in range(max_iter):
        gu = grad_nd(u_bar, bc=bc)
        for i in range(len(ps)): ps[i] = ps[i] + sigma*gu[i]
        if isotropic:
            n = xp.sqrt(sum([p*p for p in ps]) + 1e-12)
            sc = xp.maximum(1.0, n/lam)
            for i in range(len(ps)): ps[i] = ps[i]/sc
        else:
            for i in range(len(ps)): ps[i] = xp.clip(ps[i], -lam, lam)
        u_new = (u + tau*(div_nd(ps, bc=bc) + f)) / (1.0 + tau)
        u_bar = u_new + theta*(u_new - u); u = u_new
    return u

def tv_chambolle_projection(f, lam=0.1, max_iter=200, bc='neumann', isotropic=True):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    p = [xp.zeros_like(f) for _ in range(f.ndim)]; tau = 0.25
    for _ in range(max_iter):
        u = f - lam*div_nd(p, bc=bc); gu = grad_nd(u, bc=bc)
        if isotropic:
            n = xp.sqrt(sum([(pi + (tau/lam)*gi)**2 for pi,gi in zip(p,gu)]) + 1e-12)
            for i in range(len(p)): p[i] = (p[i] + (tau/lam)*gu[i]) / xp.maximum(1.0, n)
        else:
            for i in range(len(p)):
                p[i] = xp.clip(p[i] + (tau/lam)*gu[i], -1.0, 1.0)
    return u

def tv_split_bregman(f, lam=0.1, mu=2.0, max_iter=60, inner_iter=2, isotropic=True, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u = f.copy(); d=[xp.zeros_like(f) for _ in range(f.ndim)]; b=[xp.zeros_like(f) for _ in range(f.ndim)]
    for _ in range(max_iter):
        rhs = f + mu*div_nd([di - bi for di,bi in zip(d,b)], bc=bc)
        for _ in range(inner_iter):
            lap = div_nd(grad_nd(u, bc=bc), bc=bc)
            u = (rhs + mu*lap) / (1.0 + mu*2.0*f.ndim)
        gu = grad_nd(u, bc=bc)
        if isotropic:
            n = xp.sqrt(sum([(gi+bi)**2 for gi,bi in zip(gu,b)]) + 1e-12)
            sh = xp.maximum(0.0, n - lam/mu) / (n + 1e-12)
            for i in range(len(d)): d[i] = (gu[i] + b[i])*sh
        else:
            for i in range(len(d)):
                w = gu[i] + b[i]
                d[i] = xp.sign(w) * xp.maximum(xp.abs(w) - lam/mu, 0.0)
        for i in range(len(b)): b[i] = b[i] + (gu[i] - d[i])
    return u

def l2_gradient_fft(f, alpha=0.5):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    den = fft_denominator(f.shape, alpha, xp) + 1e-8
    F = xp.fft.fftn(f); U = F/den
    return xp.fft.ifftn(U).real.astype(f.dtype, copy=False)

def l3_gradient_descent(f, alpha=0.05, step=0.12, max_iter=300, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u = f.copy(); eps = xp.asarray(1e-6, dtype=f.dtype)
    for _ in range(max_iter):
        gu = grad_nd(u, bc=bc); mag = xp.sqrt(sum([g*g for g in gu]) + eps)
        reg = div_nd([3.0*mag*g for g in gu], bc=bc)
        u = u - step*((u - f) - alpha*reg)
    return u

def l0_gradient_minimization(f, lam=0.04, beta_rate=2.0, beta_max=2e4, iters_per_beta=2):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u = f.copy(); beta = 2.0*lam; shape=f.shape
    Dx2 = fft_freq_symbol(shape, xp); S = Dx2[0]
    for g in Dx2[1:]: S = S + g
    one = xp.ones(shape, dtype=f.dtype)
    while beta < beta_max:
        for _ in range(iters_per_beta):
            gu = grad_nd(u, bc='periodic'); t = sum([g*g for g in gu])
            mask = t < (lam/beta); h = [g.copy() for g in gu]
            for i in range(len(h)): h[i][mask] = 0.0
            Ff = xp.fft.fftn(f); rhs = Ff.copy()
            for axis, h_axis in enumerate(h):
                n=shape[axis]; k=xp.arange(n, dtype=xp.float32); w=2.0*xp.pi*k/float(n)
                C = (xp.exp(1j*w) - 1.0).reshape([n if i==axis else 1 for i in range(f.ndim)])
                rhs += beta * xp.conj(C) * xp.fft.fftn(h_axis)
            den = one + beta*S + 1e-8
            u = xp.fft.ifftn(rhs/den).real.astype(f.dtype, copy=False)
        beta *= beta_rate
    return u

# --- TGV^2 (single-λ) ---
def _pairs_d(d): return [(i,j) for i in range(d) for j in range(i,d)]
def _sym_grad(v_list, bc='neumann'):
    d=len(v_list); G=[[None]*d for _ in range(d)]
    for j in range(d):
        gj = grad_nd(v_list[j], bc=bc)
        for i in range(d): G[i][j]=gj[i]
    E={}
    for i in range(d):
        for j in range(i,d):
            E[(i,j)] = G[i][j] if i==j else 0.5*(G[i][j]+G[j][i])
    return E
def _div_sym(Q, bc='neumann'):
    d = max(k for pair in Q for k in pair)+1
    r=[]
    for j in range(d):
        per=[Q[(i,j)] if i<=j else Q[(j,i)] for i in range(d)]
        r.append(div_nd(per, bc=bc))
    return r
def _proj_iso_vector(p_list, alpha):
    xp = cp.get_array_module(p_list[0]) if (cp is not None and isinstance(p_list[0], cp.ndarray)) else np
    n = xp.sqrt(sum([p*p for p in p_list]) + 1e-12)
    sc = xp.maximum(1.0, n/alpha)
    for i in range(len(p_list)): p_list[i] = p_list[i]/sc
    return p_list
def _proj_iso_sym(Q, alpha):
    anyk = next(iter(Q))
    xp = cp.get_array_module(Q[anyk]) if (cp is not None and isinstance(Q[anyk], cp.ndarray)) else np
    nsq=None
    for (i,j),q in Q.items():
        w = 1.0 if (i==j) else math.sqrt(2.0)
        term = (w*q)*(w*q)
        nsq = term if nsq is None else (nsq+term)
    n = xp.sqrt(nsq + 1e-12); sc = xp.maximum(1.0, n/alpha)
    for k in Q: Q[k] = Q[k]/sc
    return Q
def tgv2_primal_dual(f, lam=0.1, max_iter=220, bc='neumann', theta=1.0):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    d=f.ndim; alpha1=lam; alpha0=2.0*lam
    u=f.copy(); u_bar=u.copy()
    v=[xp.zeros_like(f) for _ in range(d)]; v_bar=[x.copy() for x in v]
    p=[xp.zeros_like(f) for _ in range(d)]; q={pair: xp.zeros_like(f) for pair in _pairs_d(d)}
    L2_est = 8.0*d; tau = 0.5/math.sqrt(L2_est); sigma = 0.5/math.sqrt(L2_est)
    for _ in range(max_iter):
        gu = grad_nd(u_bar, bc=bc)
        for i in range(d): p[i] = p[i] + sigma*(gu[i] - v_bar[i])
        _proj_iso_vector(p, alpha1)
        Ev = _sym_grad(v_bar, bc=bc)
        for k in q: q[k] = q[k] + sigma*Ev[k]
        _proj_iso_sym(q, alpha0)
        divp = div_nd(p, bc=bc)
        u_new = (u + tau*(divp + f)) / (1.0 + tau)
        divsym = _div_sym(q, bc=bc)
        v_new = [v[i] + tau*(p[i] - divsym[i]) for i in range(d)]
        u_bar = u_new + theta*(u_new - u)
        v_bar = [v_new[i] + theta*(v_new[i] - v[i]) for i in range(d)]
        u,v = u_new, v_new
    return u

# --- PnP-ADMM with TV prox (single-λ) ---
def pnp_admm_tv(f, lam=0.1, outer_iter=40, rho=1.0, denoise_iter=25, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    x=f.copy(); z=f.copy(); u=xp.zeros_like(f)
    for _ in range(outer_iter):
        x = (f + rho*(z - u)) / (1.0 + rho)
        z = tv_primal_dual(x + u, lam=lam/rho, max_iter=denoise_iter, bc=bc)
        u = u + x - z
    return z


# First-Order Penalties — Charbonnier / Huber / p-Norm / Robust M-Estimators

**Models**  
- **Charbonnier / pseudo-Huber**: φ(t)=√(t²+ε²) − ε (smooth TV).  
- **Huber TV (anisotropic prox)**: quadratic near 0, linear for large |t|.  
- **p-norm TV**: ∑|∇u|^p, p∈(0,2]; p<1 is nonconvex → IRLS.  
- **Robust M-estimators**: Welsch, Geman-McClure (nonconvex) via IRLS.

**Solvers**  
Iteratively reweighted quadratic/TV: update weights w(|∇u|) and solve
`(I - μ·div(w ∇u)) u = rhs` with a few Jacobi sweeps (Neumann BC). Stable, GPU-friendly.

**Key parameters**  
- `lam` (strength), `iters` (outer IRLS), `inner_iter` (Jacobi sweeps), `eps/delta/scale/p`.


In [7]:
# === First-Order (robust) penalties ===
def _diffusion_step(u, f, w, mu=1.0, inner_iter=2, bc='neumann'):
    # solves (I - mu div(w ∇u)) u ≈ f with a few Jacobi iterations
    xp = cp.get_array_module(u) if (cp is not None and isinstance(u, cp.ndarray)) else np
    rhs = f
    for _ in range(inner_iter):
        gu = grad_nd(u, bc=bc)
        wg = [w*gi for gi in gu]
        lap = div_nd(wg, bc=bc)
        u = (rhs + mu*lap) / (1.0 + mu*2.0*u.ndim)
    return u

def charbonnier_tv(f, lam=0.1, eps=1e-3, iters=10, inner_iter=2, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u = f.copy()
    for _ in range(iters):
        g = grad_nd(u, bc=bc); mag = xp.sqrt(sum([gi*gi for gi in g]) + eps*eps)
        w = lam / xp.maximum(mag, eps)  # lagged diffusivity
        u = _diffusion_step(u, f, w, mu=1.0, inner_iter=inner_iter, bc=bc)
    return u

def huber_tv_aniso_sb(f, lam=0.1, delta=0.02, mu=2.0, max_iter=50, inner_iter=2, bc='neumann'):
    # Split-Bregman with component-wise Huber prox
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u=f.copy(); d=[xp.zeros_like(f) for _ in range(f.ndim)]; b=[xp.zeros_like(f) for _ in range(f.ndim)]
    lam_mu = lam/mu
    for _ in range(max_iter):
        rhs = f + mu*div_nd([di - bi for di,bi in zip(d,b)], bc=bc)
        for _ in range(inner_iter):
            lap = div_nd(grad_nd(u, bc=bc), bc=bc)
            u = (rhs + mu*lap) / (1.0 + mu*2.0*f.ndim)
        gu = grad_nd(u, bc=bc)
        for i in range(len(d)):
            w = gu[i] + b[i]
            # Huber prox_{lam/mu * huber_delta}(w):
            # if |w| <= lam -> quadratic branch: w/(1+lam/mu/delta)
            # else -> soft-threshold by lam
            absw = xp.abs(w)
            quad = absw <= lam
            d[i] = xp.where(quad, w/(1.0 + lam_mu/delta),
                            xp.sign(w)*xp.maximum(absw - lam_mu, 0.0))
        for i in range(len(b)): b[i] = b[i] + (gu[i] - d[i])
    return u

def pnorm_tv_irls(f, lam=0.1, p=0.8, eps=1e-4, iters=12, inner_iter=2, bc='neumann'):
    # p in (0,2]; p<1 nonconvex (use IRLS)
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u=f.copy()
    for _ in range(iters):
        g = grad_nd(u, bc=bc); mag2 = sum([gi*gi for gi in g]) + eps
        w = lam * (p/2.0) * mag2**(p/2.0 - 1.0)
        u = _diffusion_step(u, f, w, mu=1.0, inner_iter=inner_iter, bc=bc)
    return u

def robust_tv_irls(f, lam=0.1, kind='welsch', scale=0.1, iters=10, inner_iter=2, bc='neumann'):
    # kind: 'welsch' (exp), 'geman' (gm)
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    s2 = scale*scale; u=f.copy()
    for _ in range(iters):
        g = grad_nd(u, bc=bc); mag2 = sum([gi*gi for gi in g])
        if kind=='welsch':
            # φ(t)=s^2/2 (1 - exp(-t^2/s^2)) -> w = exp(-|∇u|^2/s^2)
            w = lam * xp.exp(-(mag2)/(s2))
        else:
            # Geman–McClure: φ(t)=t^2/(t^2+s^2) -> w = s^2/(|∇u|^2 + s^2)^2 (up to const)
            w = lam * (s2 / (mag2 + s2))
        u = _diffusion_step(u, f, w, mu=1.0, inner_iter=inner_iter, bc=bc)
    return u


# Anisotropic First-Order — Structure-Tensor / Diffusion-Tensor TV, Directional TV (dTV)

**Structure/Diffusion-Tensor TV**  
Guided smoothing along structures; reduce smoothing **across** edges. We use a simple guidance: weights per voxel  
`w = 1 / (1 + (‖∇(avg_filtered(f))‖ / s)²)` and solve `(I − div(w ∇u))u ≈ f` (few Jacobi steps).

**Directional TV (dTV)**  
Given a **unit direction field** `d(x)` (from smoothed gradient of f), penalize perpendicular gradients stronger than parallel:  
project dual onto an **elliptical ball** with radii `(λ_perp, λ_par)` via per-voxel decomposition.

**Parameters**  
- `lam_perp`, `lam_par` (dTV), `sigma_grad` (guidance smoothing), `iters/inner_iter`.


In [8]:
# === Anisotropic First-Order ===
def tensor_tv_guided(f, lam=0.1, sigma_grad=1, iters=8, inner_iter=2, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    # guidance from smoothed gradient magnitude
    g = grad_nd(avg_filter_nd(f, r=max(1, int(sigma_grad))), bc=bc)
    mag = xp.sqrt(sum([gi*gi for gi in g]) + 1e-12)
    s = xp.asarray(0.1, dtype=f.dtype)
    w = lam / (1.0 + (mag/s)**2)  # smaller across strong edges
    u = f.copy()
    for _ in range(iters):
        u = _diffusion_step(u, f, w, mu=1.0, inner_iter=inner_iter, bc=bc)
    return u

def _proj_elliptical_dual(p_list, dfield, lam_perp, lam_par):
    # project vector p onto ellipse: sqrt((|p_perp|/lam_perp)^2 + (|p_par|/lam_par)^2) <= 1
    p0 = p_list[0]
    xp = cp.get_array_module(p0) if (cp is not None and isinstance(p0, cp.ndarray)) else np
    # build vector field p (stack last axis)
    P = xp.stack(p_list, axis=-1)  # [..., D]
    D = P.shape[-1]
    d = dfield
    # parallel component
    dot = xp.sum(P*d, axis=-1, keepdims=True)
    p_par = dot * d
    p_perp = P - p_par
    n = xp.sqrt((xp.sum(p_perp*p_perp, axis=-1, keepdims=True)/(lam_perp*lam_perp)) +
                (xp.sum(p_par*p_par,   axis=-1, keepdims=True)/(lam_par*lam_par)) + 1e-12)
    sc = xp.maximum(1.0, n)
    Pn = P / sc
    # split back
    for i in range(D):
        p_list[i] = Pn[..., i]
    return p_list

def directional_tv(f, lam_perp=0.12, lam_par=0.04, max_iter=250, sigma_dir=1, bc='neumann'):
    # Chambolle-Pock variant with anisotropic dual projection using guidance directions
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    # direction field from smoothed gradient of f
    g = grad_nd(avg_filter_nd(f, r=max(1,int(sigma_dir))), bc=bc)
    mag = xp.sqrt(sum([gi*gi for gi in g]) + 1e-12)
    d = [gi/(mag+1e-12) for gi in g]
    dstack = xp.stack(d, axis=-1)  # unit vectors
    u=f.copy(); u_bar=u.copy(); ps=[xp.zeros_like(f) for _ in range(f.ndim)]
    L2=4.0*f.ndim; tau=math.sqrt(0.9/L2); sigma=math.sqrt(0.9/L2)
    for _ in range(max_iter):
        gu = grad_nd(u_bar, bc=bc)
        for i in range(len(ps)): ps[i] = ps[i] + sigma*gu[i]
        _proj_elliptical_dual(ps, dstack, lam_perp, lam_par)
        u_new = (u + tau*(div_nd(ps, bc=bc) + f)) / (1.0 + tau)
        u_bar = u_new + (u_new - u); u = u_new
    return u


# Higher-Order Gradient Regularizers — HOTV (k=2), Hessian-L1 (approx), Euler’s Elastica

**HOTV (k=2)**  
ℓ1 on second differences along each axis; Split-Bregman with soft-threshold on aux variables.

**Hessian-L1 (approx)**  
Sum of absolute Hessian components (anisotropic proxy to Hessian-Schatten); no SVD needed, 2D/3D friendly.

**Euler’s Elastica (curvature)**  
Explicit stable descent: `u ← u − step [ (u−f) − λ·div( κ n ) ]`, with `n = ∇u/|∇u|`, `κ = div(n)`. Nonconvex but practical.

**Parameters**  
- HOTV: `lam`, `mu`, `max_iter`  
- Hessian-L1: `lam`, `iters`, `inner_iter`  
- Elastica: `lam`, `step`, `max_iter`


In [9]:
# === Higher-Order ===
def second_diff_nd(u, bc='neumann'):
    # returns list of second differences along each axis
    xp = cp.get_array_module(u) if (cp is not None and isinstance(u, cp.ndarray)) else np
    outs=[]
    for ax in range(u.ndim):
        # D2 u ~ u[i+1] - 2u[i] + u[i-1]
        up = xp.roll(u, -1, axis=ax); um = xp.roll(u, 1, axis=ax)
        d2 = up - 2.0*u + um
        if bc=='neumann':
            sl0=[slice(None)]*u.ndim; sl0[ax]=0
            sln=[slice(None)]*u.ndim; sln[ax]=-1
            d2[tuple(sl0)] = (u[tuple(sl0)] - 2.0*u[tuple(sl0)] + u[tuple(sl0)])  # 0
            d2[tuple(sln)] = (u[tuple(sln)] - 2.0*u[tuple(sln)] + u[tuple(sln)])  # 0
        outs.append(d2)
    return outs

def hotv_k2_sb(f, lam=0.1, mu=2.0, max_iter=50, inner_iter=2, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u=f.copy(); d=[xp.zeros_like(f) for _ in range(f.ndim)]; b=[xp.zeros_like(f) for _ in range(f.ndim)]
    for _ in range(max_iter):
        rhs = f + mu*div_nd([di - bi for di,bi in zip(d,b)], bc=bc)  # using div of second-diff proxy (approx)
        for _ in range(inner_iter):
            lap = div_nd(grad_nd(u, bc=bc), bc=bc)
            u = (rhs + mu*lap) / (1.0 + mu*2.0*f.ndim)  # simple smoother
        D2 = second_diff_nd(u, bc=bc)
        for i in range(len(d)):
            w = D2[i] + b[i]
            d[i] = cp.sign(w)*cp.maximum(cp.abs(w) - lam/mu, 0.0) if (cp is not None and isinstance(f, cp.ndarray)) \
                   else np.sign(w)*np.maximum(np.abs(w) - lam/mu, 0.0)
        for i in range(len(b)): b[i] = b[i] + (D2[i] - d[i])
    return u

def hessian_components(u, bc='neumann'):
    # approximate Hessian via first-order grads of grads
    g = grad_nd(u, bc=bc)
    H = []
    for i in range(len(g)):
        gg = grad_nd(g[i], bc=bc)  # ∂i∂j
        H.append(gg)
    # H[i][j] = d/dx_j (∂i u)
    return H  # nested list

def hessian_l1_approx(f, lam=0.08, iters=8, inner_iter=2, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u=f.copy()
    for _ in range(iters):
        H = hessian_components(u, bc=bc)
        # anisotropic L1: weight = lam / max(|H|, eps)
        eps = xp.asarray(1e-4, dtype=f.dtype)
        acc = xp.zeros_like(f)
        for i in range(len(H)):
            for j in range(len(H[i])):
                hij = H[i][j]
                w = lam / xp.maximum(xp.abs(hij), eps)
                # diffusion step re-using first-order framework
                acc += div_nd([w*hij if k==j else 0*hij for k in range(f.ndim)], bc=bc)
        # Single Jacobi-like update
        u = (f + acc) / (1.0 + 2.0*u.ndim)
        for _ in range(inner_iter-1):
            u = (f + acc) / (1.0 + 2.0*u.ndim)
    return u

def euler_elastica(f, lam=0.08, step=0.15, max_iter=200, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    u=f.copy(); eps=xp.asarray(1e-6, dtype=f.dtype)
    for _ in range(max_iter):
        g = grad_nd(u, bc=bc); mag = xp.sqrt(sum([gi*gi for gi in g]) + eps)
        n = [gi/(mag+eps) for gi in g]
        kappa = div_nd(n, bc=bc)  # curvature
        reg = div_nd([kappa*ni for ni in n], bc=bc)
        u = u - step*((u - f) - lam*reg)
    return u


# Graph-Based Gradients — Weighted Local Graph TV (2D/3D)

**Idea**  
Use **intensity-aware edge weights** between 4/6-neighbors:  
`w = exp( − (ΔI)² / (2σ²) )`. Define weighted gradient `∇_w u` and matching divergence `div_w`, then run a primal–dual TV with projection radius `λ` on the weighted dual.

**Parameters**  
- `lam`, `sigma` (controls edge weights), `max_iter`, `bc`.


In [10]:
# === Graph-based Weighted TV (local) ===
def _edge_weights_local(f, sigma=0.1, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    ws=[]
    for ax in range(f.ndim):
        diff = xp.zeros_like(f)
        sls=[slice(None)]*f.ndim; slt=[slice(None)]*f.ndim
        sls[ax]=slice(1,None); slt[ax]=slice(0,-1)
        diff[tuple(slt)] = f[tuple(sls)] - f[tuple(slt)]
        if bc=='periodic':
            last=[slice(None)]*f.ndim; last[ax]=-1
            first=[slice(None)]*f.ndim; first[ax]=0
            diff[tuple(last)] = f[tuple(first)] - f[tuple(last)]
        w = xp.exp( - (diff*diff) / (2.0*sigma*sigma) )
        ws.append(w)
    return ws

def graph_tv_weighted(f, lam=0.1, sigma=0.1, max_iter=250, bc='neumann'):
    # Primal-dual with weighted grad/div
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    w = _edge_weights_local(avg_filter_nd(f, r=1), sigma=sigma, bc=bc)
    u=f.copy(); u_bar=u.copy(); ps=[xp.zeros_like(f) for _ in range(f.ndim)]
    L2=4.0*f.ndim; tau=math.sqrt(0.9/L2); sigma_t=math.sqrt(0.9/L2)
    for _ in range(max_iter):
        # weighted grad
        gu = grad_nd(u_bar, bc=bc)
        for i in range(len(ps)):
            ps[i] = ps[i] + sigma_t*(w[i]*gu[i])
        # isotropic projection with weights
        n = xp.sqrt(sum([(pi**2) for pi in ps]) + 1e-12)
        sc = xp.maximum(1.0, n/lam)
        for i in range(len(ps)): ps[i] = ps[i]/sc
        # weighted divergence
        divp = div_nd([w[i]*ps[i] for i in range(len(ps))], bc=bc)
        u_new = (u + tau*(divp + f)) / (1.0 + tau)
        u_bar = u_new + (u_new - u); u = u_new
    return u


# Composite & Hybrid Formulations — Weighted TV (guided), Elastic TV (TV+L2)

**Weighted TV (guided)**  
Spatially varying regularization `λ(x) = λ · g(x)`, where `g(x)=1 / (1 + γ·‖∇(avg(f))‖)` to preserve strong edges.

**Elastic TV (TV + α‖∇u‖²)**  
Mix TV with a small L2 term for smoother transitions; solved by alternating a TV step and an L2 step (few rounds).

**Parameters**  
- Weighted TV: `lam`, `gamma`  
- Elastic TV: `lam`, `alpha_l2`, `rounds`


In [11]:
# === Composite & Hybrid ===
def weighted_tv_guided(f, lam=0.1, gamma=4.0, max_iter=250, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    g = grad_nd(avg_filter_nd(f, r=1), bc=bc)
    mag = xp.sqrt(sum([gi*gi for gi in g]) + 1e-12)
    w = 1.0 / (1.0 + gamma*mag)  # smaller λ near strong edges
    # primal-dual with spatial λ: project with radius lam*w
    u=f.copy(); u_bar=u.copy(); ps=[xp.zeros_like(f) for _ in range(f.ndim)]
    L2=4.0*f.ndim; tau=math.sqrt(0.9/L2); sigma=math.sqrt(0.9/L2)
    for _ in range(max_iter):
        gu = grad_nd(u_bar, bc=bc)
        for i in range(len(ps)): ps[i] = ps[i] + sigma*gu[i]
        n = xp.sqrt(sum([p*p for p in ps]) + 1e-12)
        sc = xp.maximum(1.0, n / (lam*w))
        for i in range(len(ps)): ps[i] = ps[i]/sc
        u_new = (u + tau*(div_nd(ps, bc=bc) + f)) / (1.0 + tau)
        u_bar = u_new + (u_new - u); u=u_new
    return u

def elastic_tv(f, lam=0.08, alpha_l2=0.2, rounds=3, bc='neumann'):
    u=f.copy()
    for _ in range(rounds):
        u = tv_primal_dual(u, lam=lam, max_iter=120, bc=bc)
        u = l2_gradient_fft(u, alpha=alpha_l2)
    return u


# Data-Term Variants — TV-L1 (robust) & TV-Poisson (photon-limited)

**Generic primal–dual with custom data prox**  
We solve: `min_u F(u) + λ·TV(u)` where `F` is the data term.

**TV-L1 (‖u−f‖₁ + λ·TV(u))**  
Prox: `prox_{τ‖·−f‖₁}(x) = f + shrink(x−f, τ)`.

**TV-Poisson (u − f·log u + λ·TV(u), u>0)**  
Prox has closed form: `prox_τ(x) = 0.5*(x − τ + sqrt((x − τ)² + 4τf))`.

**Parameters**  
- `lam` (TV strength); internal step sizes auto-set; `max_iter`.


In [12]:
# === Data-term variants ===
def tv_primal_dual_general(f, lam=0.1, max_iter=300, bc='neumann', prox_data=None):
    # F(u) via prox_data(u, tau); G(Ku)=λ||Ku||_1 with K=grad
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    if prox_data is None:
        # default L2: prox_{τ*0.5||u-f||^2}(x) = (x + τ f)/(1+τ)
        prox_data = lambda x,tau: (x + tau*f)/(1.0 + tau)
    u = f.copy(); u_bar=u.copy(); ps=[xp.zeros_like(f) for _ in range(f.ndim)]
    L2=4.0*f.ndim; tau = math.sqrt(0.9/L2); sigma=math.sqrt(0.9/L2)
    for _ in range(max_iter):
        gu = grad_nd(u_bar, bc=bc)
        for i in range(len(ps)): ps[i] = ps[i] + sigma*gu[i]
        n = xp.sqrt(sum([p*p for p in ps]) + 1e-12)
        sc = xp.maximum(1.0, n/lam)
        for i in range(len(ps)): ps[i] = ps[i]/sc
        # primal step with data prox
        u_new = prox_data(u + tau*div_nd(ps, bc=bc), tau)
        u_bar = u_new + (u_new - u); u = u_new
    return u

def tv_l1_data(f, lam=0.1, max_iter=300, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    def prox_l1(x, tau):
        d = x - f
        return f + (xp.sign(d) * xp.maximum(xp.abs(d) - tau, 0.0))
    return tv_primal_dual_general(f, lam=lam, max_iter=max_iter, bc=bc, prox_data=prox_l1)

def tv_poisson_data(f, lam=0.1, max_iter=300, bc='neumann'):
    xp = cp.get_array_module(f) if (cp is not None and isinstance(f, cp.ndarray)) else np
    fpos = xp.maximum(f, 0.0)
    def prox_poisson(x, tau):
        # positive proximal (avoid negative)
        return 0.5*(x - tau + xp.sqrt((x - tau)**2 + 4.0*tau*fpos))
    return tv_primal_dual_general(f, lam=lam, max_iter=max_iter, bc=bc, prox_data=prox_poisson)


# 2D UI — All Models (separate save)

**What this cell does**  
- Lets you pick **any model** (base + new), tune relevant params, run on a **single 2D image**, **preview**, and **save** to a file.

**Paths (edit as needed)**  
- Input: `/home/askiran/data/sample2d.tif`  
- Output: `/home/askiran/data/sample2d_out.tif`

**Notes**  
- Only shows parameters relevant to the chosen method.  
- GPU toggle; CPU fallback.


In [13]:
# === 2D UI (all models) ===
import ipywidgets as W
import matplotlib.pyplot as plt

img2d_w = W.Text("/home/askiran/data/test.tif", description="Image", layout=W.Layout(width="600px"))
out2d_w = W.Text("/home/askiran/data/test_out.tif", description="Save to", layout=W.Layout(width="600px"))
usegpu2d_w = W.Checkbox(value=True, description="Use GPU")

METHODS = [
    "TV (L1)", "TGV (2nd order)", "L0", "L2", "L3", "PnP-ADMM (TV)",
    "Charbonnier TV", "Huber TV", "p-norm TV", "Robust TV (Welsch)", "Robust TV (Geman-McClure)",
    "Tensor TV (structure-guided)", "Directional TV (dTV)",
    "HOTV (k=2)", "Hessian L1 (approx)", "Euler Elastica",
    "Graph-TV", "Weighted TV (guided)",
    "TV-L1 data fidelity", "TV-Poisson data fidelity"
]
method2d_w = W.Dropdown(options=METHODS, value="TGV (2nd order)", description="Method")

# common params
lam_w = W.FloatLogSlider(value=0.12, base=10, min=-3.0, max=0.5, step=0.05, description="lambda")
alpha_w = W.FloatLogSlider(value=0.6, base=10, min=-3.0, max=1.0, step=0.05, description="alpha/L2")
step_w  = W.FloatSlider(value=0.12, min=0.01, max=0.3, step=0.01, description="step")
maxit_w = W.IntSlider(value=250, min=20, max=1500, step=10, description="max_iter")
bc_w    = W.Dropdown(options=["neumann", "periodic"], value="neumann", description="BC")

# special params
eps_w   = W.FloatLogSlider(value=1e-3, base=10, min=-6, max=-1, step=0.1, description="epsilon")
delta_w = W.FloatLogSlider(value=2e-2, base=10, min=-4, max=-1, step=0.1, description="delta (Huber)")
p_w     = W.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.05, description="p (p-norm)")
scale_w = W.FloatLogSlider(value=0.1, base=10, min=-3, max=0, step=0.1, description="scale (robust)")
sigma_w = W.FloatLogSlider(value=0.1, base=10, min=-3, max=0, step=0.1, description="sigma (graph)")
gamma_w = W.FloatLogSlider(value=4.0, base=10, min=-1, max=2, step=0.05, description="gamma (guided)")
lam_perp_w = W.FloatLogSlider(value=0.12, base=10, min=-3, max=0.5, step=0.05, description="λ_perp (dTV)")
lam_par_w  = W.FloatLogSlider(value=0.04, base=10, min=-3, max=0.5, step=0.05, description="λ_par (dTV)")
sigma_dir_w= W.IntSlider(value=1, min=1, max=5, step=1, description="sigma_dir")
sigma_grad_w= W.IntSlider(value=1, min=1, max=5, step=1, description="sigma_grad")

run2d_btn = W.Button(description="Run 2D", button_style="success")
out2d_area = W.Output()

def _toggle2d(_=None):
    m = method2d_w.value
    # Hide/show
    lam_w.layout.display = 'flex' if m not in ('L2', 'L3') else 'flex'
    alpha_w.layout.display = 'flex' if m in ('L2','L3','elastic tv (not used here)') else 'none'
    step_w.layout.display  = 'flex' if m in ('L3','Euler Elastica') else 'none'
    eps_w.layout.display   = 'flex' if m == 'Charbonnier TV' else 'none'
    delta_w.layout.display = 'flex' if m == 'Huber TV' else 'none'
    p_w.layout.display     = 'flex' if m == 'p-norm TV' else 'none'
    scale_w.layout.display = 'flex' if 'Geman' in m or 'Welsch' in m else 'none'
    sigma_w.layout.display = 'flex' if m == 'Graph-TV' else 'none'
    gamma_w.layout.display = 'flex' if m == 'Weighted TV (guided)' else 'none'
    lam_perp_w.layout.display = 'flex' if m == 'Directional TV (dTV)' else 'none'
    lam_par_w.layout.display  = 'flex' if m == 'Directional TV (dTV)' else 'none'
    sigma_dir_w.layout.display= 'flex' if m == 'Directional TV (dTV)' else 'none'
    sigma_grad_w.layout.display= 'flex' if m == 'Tensor TV (structure-guided)' else 'none'
    # max_iter generally relevant; keep visible for most
    maxit_w.layout.display = 'flex'
_toggle2d()
method2d_w.observe(_toggle2d, names='value')

def _run2d(_):
    out2d_area.clear_output()
    with out2d_area:
        xp = get_xp(usegpu2d_w.value)
        f = read_image_2d(img2d_w.value, use_gpu=usegpu2d_w.value, dtype='float32')
        print("Loaded:", img2d_w.value, "| shape:", to_cpu(f).shape, "| backend:", "CuPy" if xp is cp else "NumPy")
        m = method2d_w.value; t0 = time.time()
        if m == "TV (L1)":
            u = tv_primal_dual(f, lam=float(lam_w.value), max_iter=int(maxit_w.value), bc=bc_w.value)
        elif m == "TGV (2nd order)":
            u = tgv2_primal_dual(f, lam=float(lam_w.value), max_iter=int(maxit_w.value), bc=bc_w.value)
        elif m == "L0":
            u = l0_gradient_minimization(f, lam=float(lam_w.value))
        elif m == "L2":
            u = l2_gradient_fft(f, alpha=float(alpha_w.value))
        elif m == "L3":
            u = l3_gradient_descent(f, alpha=float(alpha_w.value), step=float(step_w.value), max_iter=int(maxit_w.value), bc=bc_w.value)
        elif m == "PnP-ADMM (TV)":
            u = pnp_admm_tv(f, lam=float(lam_w.value), outer_iter=min(60, int(maxit_w.value)), bc=bc_w.value)
        elif m == "Charbonnier TV":
            u = charbonnier_tv(f, lam=float(lam_w.value), eps=float(eps_w.value), iters=min(20,int(maxit_w.value//20)+8), bc=bc_w.value)
        elif m == "Huber TV":
            u = huber_tv_aniso_sb(f, lam=float(lam_w.value), delta=float(delta_w.value), max_iter=min(80,int(maxit_w.value)), bc=bc_w.value)
        elif m == "p-norm TV":
            u = pnorm_tv_irls(f, lam=float(lam_w.value), p=float(p_w.value), iters=min(20,int(maxit_w.value//20)+8), bc=bc_w.value)
        elif m == "Robust TV (Welsch)":
            u = robust_tv_irls(f, lam=float(lam_w.value), kind='welsch', scale=float(scale_w.value), iters=min(20,int(maxit_w.value//20)+8), bc=bc_w.value)
        elif m == "Robust TV (Geman-McClure)":
            u = robust_tv_irls(f, lam=float(lam_w.value), kind='geman', scale=float(scale_w.value), iters=min(20,int(maxit_w.value//20)+8), bc=bc_w.value)
        elif m == "Tensor TV (structure-guided)":
            u = tensor_tv_guided(f, lam=float(lam_w.value), sigma_grad=int(sigma_grad_w.value), iters=min(20,int(maxit_w.value//20)+8), bc=bc_w.value)
        elif m == "Directional TV (dTV)":
            u = directional_tv(f, lam_perp=float(lam_perp_w.value), lam_par=float(lam_par_w.value), max_iter=int(maxit_w.value), sigma_dir=int(sigma_dir_w.value), bc=bc_w.value)
        elif m == "HOTV (k=2)":
            u = hotv_k2_sb(f, lam=float(lam_w.value), max_iter=min(80,int(maxit_w.value)), bc=bc_w.value)
        elif m == "Hessian L1 (approx)":
            u = hessian_l1_approx(f, lam=float(lam_w.value), iters=min(20,int(maxit_w.value//20)+8), bc=bc_w.value)
        elif m == "Euler Elastica":
            u = euler_elastica(f, lam=float(lam_w.value), step=float(step_w.value), max_iter=int(maxit_w.value), bc=bc_w.value)
        elif m == "Graph-TV":
            u = graph_tv_weighted(f, lam=float(lam_w.value), sigma=float(sigma_w.value), max_iter=int(maxit_w.value), bc=bc_w.value)
        elif m == "Weighted TV (guided)":
            u = weighted_tv_guided(f, lam=float(lam_w.value), gamma=float(gamma_w.value), max_iter=int(maxit_w.value), bc=bc_w.value)
        elif m == "TV-L1 data fidelity":
            u = tv_l1_data(f, lam=float(lam_w.value), max_iter=int(maxit_w.value), bc=bc_w.value)
        elif m == "TV-Poisson data fidelity":
            u = tv_poisson_data(f, lam=float(lam_w.value), max_iter=int(maxit_w.value), bc=bc_w.value)
        else:
            raise ValueError("Unknown method")
        dt = time.time()-t0
        u_cpu = to_cpu(u); f_cpu = to_cpu(f)
        iio.imwrite(out2d_w.value, np.clip(u_cpu, 0, 1))
        print(f"Saved: {out2d_w.value} | time {dt:.2f}s")
        fig,(ax1,ax2)=plt.subplots(1,2, figsize=(10,5))
        ax1.imshow(f_cpu, cmap='gray'); ax1.set_title("Input"); ax1.axis('off')
        ax2.imshow(u_cpu, cmap='gray'); ax2.set_title(f"{m}"); ax2.axis('off')
        plt.show()

run2d_btn.on_click(_run2d)
W.VBox([
    W.HBox([img2d_w]), W.HBox([out2d_w]),
    W.HBox([method2d_w, usegpu2d_w]),
    W.HBox([lam_w, alpha_w, step_w]),
    W.HBox([maxit_w, bc_w]),
    W.HBox([eps_w, delta_w, p_w, scale_w]),
    W.HBox([sigma_w, gamma_w]),
    W.HBox([lam_perp_w, lam_par_w, sigma_dir_w, sigma_grad_w]),
    run2d_btn, out2d_area
])


In [14]:
# === 3D UI (all models) ===
in3d_w  = W.Text("/home/askiran/data/Peri_1/", description="Input dir", layout=W.Layout(width="600px"))
out3d_w = W.Text("/home/askiran/data/stack3d_out/", description="Output dir", layout=W.Layout(width="600px"))
usegpu3d_w = W.Checkbox(value=True, description="Use GPU")
method3d_w = W.Dropdown(options=METHODS, value="TGV (2nd order)", description="Method")
lam3_w = W.FloatLogSlider(value=0.10, base=10, min=-3.0, max=0.5, step=0.05, description="lambda")
alpha3_w = W.FloatLogSlider(value=0.6, base=10, min=-3.0, max=1.0, step=0.05, description="alpha/L2")
step3_w  = W.FloatSlider(value=0.10, min=0.01, max=0.3, step=0.01, description="step")
maxit3_w = W.IntSlider(value=220, min=20, max=1500, step=10, description="max_iter")
bc3_w    = W.Dropdown(options=["neumann", "periodic"], value="neumann", description="BC")
slab_w   = W.IntText(value=0, description="Z slab (0=full)")

# Special params reuse
eps3_w, delta3_w, p3_w = eps_w, delta_w, p_w
scale3_w, sigma3_w, gamma3_w = scale_w, sigma_w, gamma_w
lam_perp3_w, lam_par3_w, sigma_dir3_w, sigma_grad3_w = lam_perp_w, lam_par_w, sigma_dir_w, sigma_grad_w

run3d_btn = W.Button(description="Run 3D", button_style="success")
out3d_area = W.Output()

def _toggle3d(_=None):
    m=method3d_w.value
    step3_w.layout.display  = 'flex' if m in ('L3','Euler Elastica') else 'none'
    alpha3_w.layout.display = 'flex' if m=='L2' else 'none'
    eps3_w.layout.display   = 'flex' if m=='Charbonnier TV' else 'none'
    delta3_w.layout.display = 'flex' if m=='Huber TV' else 'none'
    p3_w.layout.display     = 'flex' if m=='p-norm TV' else 'none'
    scale3_w.layout.display = 'flex' if 'Geman' in m or 'Welsch' in m else 'none'
    sigma3_w.layout.display = 'flex' if m=='Graph-TV' else 'none'
    gamma3_w.layout.display = 'flex' if m=='Weighted TV (guided)' else 'none'
    lam_perp3_w.layout.display = 'flex' if m=='Directional TV (dTV)' else 'none'
    lam_par3_w.layout.display  = 'flex' if m=='Directional TV (dTV)' else 'none'
    sigma_dir3_w.layout.display= 'flex' if m=='Directional TV (dTV)' else 'none'
    sigma_grad3_w.layout.display= 'flex' if m=='Tensor TV (structure-guided)' else 'none'
    # slab disabled for global FFT solvers
    slab_w.disabled = (m in ('L0','L2'))
_toggle3d()
method3d_w.observe(_toggle3d, names='value')

def _run3d(_):
    out3d_area.clear_output()
    with out3d_area:
        os.makedirs(out3d_w.value, exist_ok=True)
        xp = get_xp(usegpu3d_w.value)
        F = read_stack_3d(in3d_w.value, use_gpu=usegpu3d_w.value, dtype='float32')
        Z,Y,X = to_cpu(F).shape
        print("Loaded:", in3d_w.value, "| shape:", (Z,Y,X), "| backend:", "CuPy" if xp is cp else "NumPy")
        m = method3d_w.value
        def run_full(vol):
            if m == "TV (L1)":
                return tv_primal_dual(vol, lam=float(lam3_w.value), max_iter=int(maxit3_w.value), bc=bc3_w.value)
            if m == "TGV (2nd order)":
                return tgv2_primal_dual(vol, lam=float(lam3_w.value), max_iter=int(maxit3_w.value), bc=bc3_w.value)
            if m == "L0":
                return l0_gradient_minimization(vol, lam=float(lam3_w.value))
            if m == "L2":
                return l2_gradient_fft(vol, alpha=float(alpha3_w.value))
            if m == "L3":
                return l3_gradient_descent(vol, alpha=float(alpha3_w.value), step=float(step3_w.value), max_iter=int(maxit3_w.value), bc=bc3_w.value)
            if m == "PnP-ADMM (TV)":
                return pnp_admm_tv(vol, lam=float(lam3_w.value), outer_iter=min(50,int(maxit3_w.value)), bc=bc3_w.value)
            if m == "Charbonnier TV":
                return charbonnier_tv(vol, lam=float(lam3_w.value), eps=float(eps3_w.value), iters=min(18,int(maxit3_w.value//20)+8), bc=bc3_w.value)
            if m == "Huber TV":
                return huber_tv_aniso_sb(vol, lam=float(lam3_w.value), delta=float(delta3_w.value), max_iter=min(80,int(maxit3_w.value)), bc=bc3_w.value)
            if m == "p-norm TV":
                return pnorm_tv_irls(vol, lam=float(lam3_w.value), p=float(p3_w.value), iters=min(18,int(maxit3_w.value//20)+8), bc=bc3_w.value)
            if m == "Robust TV (Welsch)":
                return robust_tv_irls(vol, lam=float(lam3_w.value), kind='welsch', scale=float(scale3_w.value), iters=min(18,int(maxit3_w.value//20)+8), bc=bc3_w.value)
            if m == "Robust TV (Geman-McClure)":
                return robust_tv_irls(vol, lam=float(lam3_w.value), kind='geman', scale=float(scale3_w.value), iters=min(18,int(maxit3_w.value//20)+8), bc=bc3_w.value)
            if m == "Tensor TV (structure-guided)":
                return tensor_tv_guided(vol, lam=float(lam3_w.value), sigma_grad=int(sigma_grad3_w.value), iters=min(18,int(maxit3_w.value//20)+8), bc=bc3_w.value)
            if m == "Directional TV (dTV)":
                return directional_tv(vol, lam_perp=float(lam_perp3_w.value), lam_par=float(lam_par3_w.value), max_iter=int(maxit3_w.value), sigma_dir=int(sigma_dir3_w.value), bc=bc3_w.value)
            if m == "HOTV (k=2)":
                return hotv_k2_sb(vol, lam=float(lam3_w.value), max_iter=min(80,int(maxit3_w.value)), bc=bc3_w.value)
            if m == "Hessian L1 (approx)":
                return hessian_l1_approx(vol, lam=float(lam3_w.value), iters=min(18,int(maxit3_w.value//20)+8), bc=bc3_w.value)
            if m == "Euler Elastica":
                return euler_elastica(vol, lam=float(lam3_w.value), step=float(step3_w.value), max_iter=int(maxit3_w.value), bc=bc3_w.value)
            if m == "Graph-TV":
                return graph_tv_weighted(vol, lam=float(lam3_w.value), sigma=float(sigma3_w.value), max_iter=int(maxit3_w.value), bc=bc3_w.value)
            if m == "Weighted TV (guided)":
                return weighted_tv_guided(vol, lam=float(lam3_w.value), gamma=float(gamma3_w.value), max_iter=int(maxit3_w.value), bc=bc3_w.value)
            if m == "TV-L1 data fidelity":
                return tv_l1_data(vol, lam=float(lam3_w.value), max_iter=int(maxit3_w.value), bc=bc3_w.value)
            if m == "TV-Poisson data fidelity":
                return tv_poisson_data(vol, lam=float(lam3_w.value), max_iter=int(maxit3_w.value), bc=bc3_w.value)
            raise ValueError("Unknown method")

        t0=time.time()
        slab = int(slab_w.value)
        if slab<=0 or method3d_w.value in ('L0','L2'):
            U = run_full(F)
        else:
            outs=[]; 
            for z0 in range(0, Z, slab):
                z1 = min(Z, z0+slab)
                outs.append(run_full(F[z0:z1]))
            U = get_xp(usegpu3d_w.value).concatenate(outs, axis=0)
        dt=time.time()-t0
        Uc = to_cpu(U)
        print(f"Computed in {dt:.2f}s. Saving {Uc.shape[0]} slices -> {out3d_w.value}")
        for z in range(Uc.shape[0]):
            iio.imwrite(os.path.join(out3d_w.value, f"slice_{z:04d}.tif"), np.clip(Uc[z], 0, 1))
        print("Done.")
        zi=Uc.shape[0]//2
        fig,(ax1,ax2)=plt.subplots(1,2, figsize=(10,5))
        ax1.imshow(to_cpu(F[zi]), cmap='gray'); ax1.set_title(f"Input Z={zi}"); ax1.axis('off')
        ax2.imshow(Uc[zi], cmap='gray'); ax2.set_title(f"{m} Z={zi}"); ax2.axis('off')
        plt.show()

run3d_btn.on_click(_run3d)
W.VBox([
    W.HBox([in3d_w]), W.HBox([out3d_w]),
    W.HBox([method3d_w, usegpu3d_w]),
    W.HBox([lam3_w, alpha3_w, step3_w]),
    W.HBox([maxit3_w, bc3_w, slab_w]),
    W.HBox([eps3_w, delta3_w, p3_w, scale3_w]),
    W.HBox([sigma3_w, gamma3_w]),
    W.HBox([lam_perp3_w, lam_par3_w, sigma_dir3_w, sigma_grad3_w]),
    run3d_btn, out3d_area
])


# Notes & Suggestions — Practical Tips for This Notebook

## 1) Parameter tuning (quick cheats)
- **TV / L1**: start `λ = 0.05–0.15`. If edges soften → lower `λ`. If grain remains → raise `λ`.
- **TGV²**: start `λ = 0.05–0.15` (internally α₁=λ, α₀=2λ). Use TGV when TV “staircases”.
- **L0**: start `λ = 0.03–0.06`. Increase for stronger piecewise-constant smoothing. Expect periodic BC wrap.
- **L2**: start `α = 0.3–1.0`. Larger = blurrier; fastest for mild denoising.
- **L3**: start `α = 0.03–0.1`, `step = 0.08–0.12`. If unstable, reduce `step`.
- **PnP-ADMM (TV)**: start `λ = 0.08–0.15`. If too strong, lower `λ`. (Internals keep ρ=1, modest inner TV iters.)
- **Robust / p-norm**: begin with Charbonnier (ε≈1e−3) or p≈0.8. Increase outer IRLS iters only if needed.

## 2) Boundary conditions (BC) — choose wisely
- **Neumann (replicated)**: best for most microscopy images; avoids wrap seams.
- **Periodic**: required by FFT-based solvers (L0/L2). If border artifacts appear, prefer TV/TGV/PnP.

## 3) Performance & memory on RTX 5080 (16 GB)
- Keep everything **float32**; avoid float64.
- **VRAM budgeting (rule of thumb)**: each full-volume float32 array ≈ `4 * Z * Y * X` bytes.
  - TV-like methods keep ~4–8 arrays concurrently (u, duals, temps). Plan for **~8×** the volume size.
  - Example: `512×1024×1024` → one array ≈ 2 GB; TV may touch ~16 GB (borderline). Use **Z-slabs** (e.g., 64–128).
- **Z-slabbing**: for TV/TGV/PnP/L3/HOTV/robust variants, set `Z slab` so that `arrays_per_algo * 4 * slab * Y * X < 12–14 GB`.
  - Start with slab=64; increase if VRAM allows.
- **FFT** (L0/L2) is **global**; cannot slab trivially. If OOM: disable GPU or switch method.
- **CuPy memory pool** helps reuse allocations automatically; long runs benefit from a single kernel/session.

## 4) FFT best practices
- Periodic solvers (L0/L2): pad/crop if borders matter; otherwise accept wrap.
- Large sizes: (slight) speedup when dimensions have small prime factors. If convenient, pad to 8/16 multiples.
- Consider `rfftn/irfftn` for real data to cut FFT memory in half (kept simple here for portability).

## 5) Stability of primal–dual steps
- We use conservative automatic steps (`τ, σ`) with `τ·σ·‖K‖² < 1` heuristics.
- If a method **oscillates or diverges**: reduce `τ` (or increase `max_iter`) or lower `λ/α`.

## 6) Guidance-driven anisotropy (tensor/directional TV)
- **Structure-tensor / diffusion-tensor TV**: good on fibers/vessels; please tune `sigma_grad` (1–3).
- **Directional TV (dTV)**: `λ_⊥` controls across-edge smoothing; `λ_∥` controls along-edge smoothing. Usually `λ_⊥ > λ_∥`.

## 7) Higher-order & curvature
- **HOTV (k=2)**: reduces staircasing vs TV with modest cost; parameterize like TV (λ).
- **Hessian-L1 (approx)**: curvature-aware without SVD; if oversmoothed, reduce λ or iterations.
- **Euler’s Elastica**: excellent contour preservation; nonconvex—use smaller `step` if unstable.

## 8) Data terms
- **TV-L1** (‖u−f‖₁): use for outliers/salt-pepper; preserves contrast.
- **TV-Poisson**: photon-limited imaging; ensure inputs are nonnegative and normalized to [0,1].

## 9) I/O, preprocessing, and saving
- All loaders normalize to **[0,1] float32**; if you pass pre-normalized floats, they’re clipped to [0,1].
- For 3D, output is a **folder of slices** as requested. Use TIFF if you want lossless and add compression externally if needed.

## 10) Troubleshooting checklist
- **Weird banding** → switch from L0/L2 (periodic) to TV/TGV (Neumann).
- **Too smooth** → lower `λ/α`, or switch to Charbonnier/Huber/robust TV for better edge keeping.
- **Computation slow** → reduce `max_iter`; for TV use Primal–Dual; for 3D enable Z-slabs.
- **CUDA OOM** → disable GPU or pick TV/TGV with smaller slabs; avoid FFT methods on huge volumes.

## 11) Extensions you can add later
- **Vectorial TV / TNV** for color/multispectral stacks (joint-channel priors).
- **HOTV (k>2)** and **Hessian–Schatten** with exact per-voxel SVD in 2D.
- **Advanced PnP** with learned denoisers (e.g., DRUNet/FDnCNN) if your environment allows.

*Tip:* Keep your dataset-specific defaults in the UI cells (pre-fill paths/params) and version them per project. This will save time and ensure reproducibility across runs.


# Model Definitions — What Each Regularizer/Data Term Does

> Below are concise definitions of all models included in this notebook, with their **objective** and a short note on the **effect** you should expect on images/volumes. Here, \(u\) is the unknown clean image, \(f\) is the observed data, \(\nabla\) is the spatial gradient, and \(\text{TV}(u)\) denotes total variation. Unless stated, norms are per-voxel and aggregated over the domain.

---

## First-order gradient penalties (on \(\|\nabla u\|\))

### **TV / L1 on gradient (ROF)**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\,\text{TV}(u)\), with \(\text{TV}(u)=\sum \|\nabla u\|_2\).  
**Effect:** Strong **edge preservation** with piecewise-flat regions; can induce **staircasing** (flat plateaus).

### **Charbonnier (pseudo-Huber) TV**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\sum \phi_\varepsilon(\|\nabla u\|_2),\quad \phi_\varepsilon(t)=\sqrt{t^2+\varepsilon^2}-\varepsilon.\)  
**Effect:** Smooth, TV-like but **less staircasing**; gentle around small gradients.

### **Huber TV**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\sum h_\delta(\|\nabla u\|),\)  
\(h_\delta(t)=\begin{cases}\tfrac{t^2}{2\delta} & |t|\le \delta\\ |t|-\tfrac\delta2 & |t|>\delta\end{cases}\).  
**Effect:** Quadratic near 0 for **noise smoothing**, linear for **edge preservation**.

### **\(p\)-norm TV**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\sum \|\nabla u\|^p,\quad p\in(0,2].\)  
**Effect:**  
- \(p<1\): **Nonconvex**, sharper edges/texture retention.  
- \(p=1\): TV.  
- \(p>1\): smoother transitions, less edge-preserving.

### **Robust M-estimators on \(\|\nabla u\|\)**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\sum \rho(\|\nabla u\|)\), with robust \(\rho\):  
- **Welsch/Leclerc:** \(\rho(t)=\frac{s^2}{2}\bigl(1-e^{-t^2/s^2}\bigr)\).  
- **Geman–McClure:** \(\rho(t)=\frac{t^2}{t^2+s^2}\).  
**Effect:** **Very edge-preserving**, keeps fine details/textures; nonconvex (solved via IRLS/HQS).

---

## Anisotropic first-order (direction-aware)

### **Structure-/Diffusion-tensor TV**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\sum \|\mathbf{A}(x)\,\nabla u\|_2\), where \(\mathbf{A}(x)\) shrinks across edges more than along them.  
**Effect:** **Guided smoothing along structures** (fibers, vessels), prevents blurring across edges.

### **Directional TV (dTV)**
**Objective:** Penalize gradient components **perpendicular** vs **parallel** to a direction field \(d(x)\):  
\(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \sum \sqrt{\tfrac{\|\nabla_\perp u\|^2}{\lambda_\perp^2}+\tfrac{\|\nabla_\parallel u\|^2}{\lambda_\parallel^2}}\).  
**Effect:** Preferential smoothing **along** a known direction; preserves oriented features.

---

## Higher-order gradient regularizers

### **HOTV (k-th order, here \(k=2\))**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\sum \|D^{(2)}u\|_1\) (L1 on second differences).  
**Effect:** Reduces **staircasing** vs TV; favors **piecewise-linear** intensity profiles.

### **Hessian–Schatten / Hessian-L1 (approx)**
**Objective (proxy used here):** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\sum \|H(u)\|_{1}\) on Hessian components; (Schatten norms use SVD of Hessian).  
**Effect:** **Curvature-aware** smoothing: cleaner interiors, crisp edges, fewer block artifacts.

### **Euler’s Elastica (curvature-based)**
**Objective (one form):** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\!\int \!|\nabla u|\,\bigl(1 + \alpha\,\kappa^2\bigr)\,dx,\) where \(\kappa\) is curvature of level sets.  
**Effect:** Outstanding **contour preservation**; smooths along boundaries; nonconvex and heavier.

---

## Zero-, second-, and third-order gradient powers

### **L0 on gradient**
**Objective:** \(\displaystyle \min_u \|u-f\|_2^2 + \lambda\,\#\{x:\nabla u(x)\ne 0\}\).  
**Effect:** **Piecewise-constant** results with sharp edges; strong noise removal; periodic BC (FFT).

### **L2 on gradient (Tikhonov)**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \alpha\|\nabla u\|_2^2.\)  
**Effect:** Fast, smooths noise but can **blur edges**; closed-form via FFT (periodic).

### **L3 on gradient**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \alpha\sum \|\nabla u\|^3.\)  
**Effect:** Middle ground between L1 and stronger smoothing; preserves edges better than L2.

---

## Second-order TV

### **TGV\(^2\) (Total Generalized Variation, order 2)**
**Objective:** \(\displaystyle \min_{u,v}\ \tfrac12\|u-f\|_2^2 + \alpha_1\|\nabla u - v\|_{2,1} + \alpha_0\|E(v)\|_{2,1}\), with \(E(v)\) the symmetric gradient of \(v\).  
**Effect:** TV-like **edge preservation** **without staircasing**; favors piecewise-smooth (not just flat).

---

## Plug-and-Play / Hybrid

### **PnP-ADMM (TV denoiser)**
**Scheme:** Alternate **data step** and **TV-denoiser prox** in ADMM:  
\(x\leftarrow \arg\min_x \tfrac12\|x-f\|^2 + \tfrac{\rho}{2}\|x-(z-u)\|^2\),  
\(z\leftarrow \text{TV-prox}_{\lambda/\rho}(x+u)\), \(u\leftarrow u + x - z\).  
**Effect:** Flexible denoising with TV prior; robust and often **artifact-light**.

### **Weighted (guided) TV**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \sum \lambda(x)\,\|\nabla u\|\), with \(\lambda(x)\) **smaller near edges**.  
**Effect:** Edge-aware smoothing without hand masking; preserves salient boundaries.

### **Elastic TV (TV + L2 on \(\nabla u\))**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\,\text{TV}(u) + \alpha\|\nabla u\|_2^2.\)  
**Effect:** Mixes TV’s edge keeping with L2’s smooth transitions; reduces TV staircasing.

---

## Graph-based (local non-Euclidean neighborhoods)

### **Graph-TV (weighted local)**
**Objective:** \(\displaystyle \min_u \tfrac12\|u-f\|_2^2 + \lambda\sum \|\nabla_w u\|\), where edge weights \(w\) depend on **intensity differences** (e.g., \(w=\exp(-\Delta I^2/2\sigma^2)\)).  
**Effect:** Respects local intensity barriers; **less smoothing across edges**, more within regions.

---

## Data-term variants (pair with any gradient prior)

### **TV-L1 data fidelity**
**Objective:** \(\displaystyle \min_u \|u-f\|_1 + \lambda\,\text{TV}(u).\)  
**Effect:** **Outlier-robust** (salt-and-pepper, impulse noise), preserves contrast; edges remain crisp.

### **TV-Poisson (photon-limited)**
**Objective:** \(\displaystyle \min_{u>0}\ \sum (u - f\log u) + \lambda\,\text{TV}(u).\)  
**Effect:** Proper for **Poisson noise** (microscopy); avoids dark bias; requires \(u>0\).

---

## Notes on boundary conditions
- **Periodic**: required by FFT-based L0/L2; can introduce wrap-around artifacts at borders.  
- **Neumann (replicated)**: default for spatial solvers (TV/TGV/PnP/etc.); avoids wrap seams.

> **Rule of thumb:** For **crisp edges** with minimal artifacts, start with **TV** or **TGV\(^2\)**. For **piecewise-constant** look, try **L0**. For **smooth-but-fast** denoising, use **L2**. For **outliers**, switch the data term to **L1**; for **photon noise**, use **Poisson**.
